In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf

X_train_seq = np.load('/kaggle/input/datasets/raedmuhammed/data600/X_train_seq.npy')
X_val_seq = np.load('/kaggle/input/datasets/raedmuhammed/data600/X_val_seq.npy')
y_train = np.load('/kaggle/input/datasets/raedmuhammed/data600/y_train.npy')
y_val = np.load('/kaggle/input/datasets/raedmuhammed/data600/y_val.npy')
X_test_seq=np.load('/kaggle/input/datasets/raedmuhammed/data600/X_test_seq.npy')
y_test=np.load('/kaggle/input/datasets/raedmuhammed/data600/y_test.npy')



train_dataset = tf.data.Dataset.from_tensor_slices((X_train_seq, y_train))
train_dataset = train_dataset.cache().shuffle(len(X_train_seq)).batch(16).prefetch(8)

val_dataset = tf.data.Dataset.from_tensor_slices((X_val_seq, y_val))
val_dataset = val_dataset.cache().batch(16).prefetch(8)

I0000 00:00:1790381648.751857      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1790381648.754874      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [2]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense, Embedding, GlobalMaxPooling1D
from tensorflow.keras.callbacks import EarlyStopping

classes = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

# handle imbalance in the data by use weighted_binary_crossentropy

pos_counts = y_train.sum(axis=0)
neg_counts = y_train.shape[0] - pos_counts

# define class weights 
class_weights = np.sqrt(neg_counts / pos_counts)

def weighted_binary_crossentropy(class_weights):
    class_weights = tf.constant(class_weights, dtype=tf.float32)
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        y_true = tf.cast(y_true, tf.float32)
        bce = -(y_true * tf.math.log(y_pred) * class_weights +
                (1 - y_true) * tf.math.log(1 - y_pred))
        return tf.reduce_mean(bce)
    return loss


model = Sequential()

model.add(Embedding(100000, 120)) 
model.add((LSTM(60, activation='tanh', return_sequences=True )))
model.add(GlobalMaxPooling1D())
model.add(Dropout(0.10))
model.add(Dense(50, activation='relu'))
model.add(Dropout(0.10))

# multi label classification
model.add(Dense(6, activation='sigmoid'))


# precision/recall per class to track performance individually
per_class_metrics = []
for i, class_name in enumerate(classes):
    per_class_metrics.append(
        tf.keras.metrics.Precision(class_id=i, name=f'precision_{class_name}')
    )
    per_class_metrics.append(
        tf.keras.metrics.Recall(class_id=i, name=f'recall_{class_name}')
    )

model.compile(
    loss=weighted_binary_crossentropy(class_weights),
    optimizer='Adam',
    metrics=per_class_metrics
)

model.summary()

# early stoping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# train
history = model.fit(
    train_dataset,
    epochs=5,
    validation_data=val_dataset,
    callbacks=[early_stop]
)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
7979/7979 ━━━━━━━━━━━━━━━━━━━━ 128s 15ms/step - loss: 0.1689 - precision_identity_hate: 0.1554 - precision_insult: 0.5483 - precision_obscene: 0.6168 - precision_severe_toxic: 0.2503 - precision_threat: 0.0734 - precision_toxic: 0.7050 - recall_identity_hate: 0.6041 - recall_insult: 0.8031 - recall_obscene: 0.8447 - recall_severe_toxic: 0.8165 - recall_threat: 0.1848 - recall_toxic: 0.7506 - val_loss: 0.1390 - val_precision_identity_hate: 0.2093 - val_precision_insult: 0.5740 - val_precision_obscene: 0.6872 - val_precision_severe_toxic: 0.2991 - val_precision_threat: 0.1667 - val_precision_toxic: 0.7692 - val_recall_identity_hate: 0.6762 - val_recall_insult: 0.8254 - val_recall_obscene: 0.8645 - val_recall_severe_toxic: 0.8469 - val_recall_threat: 0.5636 - val_recall_toxic: 0.8006
Epoch 2/5
7979/7979 ━━━━━━━━━━━━━━━━━━━━ 120s 15ms/step - loss: 0.1109 - precision_identity_hate: 0.2504 - precision_insult: 0.5748 - precision_obscene: 0.7092 - precision_severe_toxic: 0.2978 - pre

In [3]:
from sklearn.metrics import precision_recall_curve, f1_score, classification_report
import numpy as np

label_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

y_val_pred_prob = model.predict(val_dataset)

best_thresholds = []

# find best threshold per class using precision-recall curve (max F1)
for i, label in enumerate(label_cols):
    precisions, recalls, thresholds = precision_recall_curve(y_val[:, i], y_val_pred_prob[:, i])
    
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
    
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    best_f1 = f1_scores[best_idx]
    
    best_thresholds.append(best_threshold)
    print(f"{label}: best_threshold={best_threshold:.3f} | precision={precisions[best_idx]:.3f} | recall={recalls[best_idx]:.3f} | F1={best_f1:.3f}")

best_thresholds = np.array(best_thresholds)
print("\nFinal thresholds (derived from validation):", best_thresholds)

# apply tuned thresholds to get final predictions
y_val_pred_tuned = np.zeros_like(y_val_pred_prob, dtype=int)
for i in range(len(label_cols)):
    y_val_pred_tuned[:, i] = (y_val_pred_prob[:, i] > best_thresholds[i]).astype(int)

f1_per_class_tuned = f1_score(y_val, y_val_pred_tuned, average=None)
print("\n=== Validation results after threshold tuning ===")
for label, score in zip(label_cols, f1_per_class_tuned):
    print(f"F1 ({label}): {score:.4f}")

f1_macro_tuned = f1_score(y_val, y_val_pred_tuned, average='macro')
f1_micro_tuned = f1_score(y_val, y_val_pred_tuned, average='micro')
print(f"\nF1 Macro (val tuned): {f1_macro_tuned:.4f}")
print(f"F1 Micro (val tuned): {f1_micro_tuned:.4f}")

print("\n", classification_report(y_val, y_val_pred_tuned, target_names=label_cols))

1995/1995 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step
toxic: best_threshold=0.707 | precision=0.791 | recall=0.780 | F1=0.785
severe_toxic: best_threshold=0.797 | precision=0.450 | recall=0.634 | F1=0.527
obscene: best_threshold=0.705 | precision=0.786 | recall=0.817 | F1=0.801
threat: best_threshold=0.749 | precision=0.442 | recall=0.418 | F1=0.430
insult: best_threshold=0.716 | precision=0.646 | recall=0.764 | F1=0.700
identity_hate: best_threshold=0.808 | precision=0.421 | recall=0.495 | F1=0.455

Final thresholds (derived from validation): [0.7068453  0.7966172  0.70450646 0.74873644 0.716241   0.80779004]

=== Validation results after threshold tuning ===
F1 (toxic): 0.7850
F1 (severe_toxic): 0.5247
F1 (obscene): 0.8006
F1 (threat): 0.4225
F1 (insult): 0.7000
F1 (identity_hate): 0.4525

F1 Macro (val tuned): 0.6142
F1 Micro (val tuned): 0.7356

                precision    recall  f1-score   support

        toxic       0.79      0.78      0.78      3059
 severe_toxic       0.45      0.63   

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [4]:

#Evaluating on the test data set

y_pred_prob = model.predict(X_test_seq, batch_size=32)

thresholds = {
    'toxic': best_thresholds[0],
    'severe_toxic': best_thresholds[1],
    'obscene': best_thresholds[2],
    'threat': best_thresholds[3],
    'insult': best_thresholds[4],
    'identity_hate':best_thresholds[5]
}

y_pred = np.zeros_like(y_pred_prob, dtype=int)
for i, label in enumerate(label_cols):
    y_pred[:, i] = (y_pred_prob[:, i] > thresholds[label]).astype(int)

f1_per_class = f1_score(y_test, y_pred, average=None)
for label, score in zip(label_cols, f1_per_class):
    print(f"F1 ({label}): {score:.4f}  (threshold={thresholds[label]})")

f1_macro = f1_score(y_test, y_pred, average='macro')
f1_micro = f1_score(y_test, y_pred, average='micro')
print(f"\nF1 Macro: {f1_macro:.4f}")
print(f"F1 Micro: {f1_micro:.4f}")

print("\n", classification_report(y_test, y_pred, target_names=label_cols))

2000/2000 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step
F1 (toxic): 0.5997  (threshold=0.7068452835083008)
F1 (severe_toxic): 0.3736  (threshold=0.7966172099113464)
F1 (obscene): 0.6064  (threshold=0.7045064568519592)
F1 (threat): 0.2986  (threshold=0.7487364411354065)
F1 (insult): 0.5547  (threshold=0.7162410020828247)
F1 (identity_hate): 0.4621  (threshold=0.8077900409698486)

F1 Macro: 0.4825
F1 Micro: 0.5735

                precision    recall  f1-score   support

        toxic       0.46      0.85      0.60      6090
 severe_toxic       0.25      0.71      0.37       367
      obscene       0.49      0.80      0.61      3691
       threat       0.26      0.35      0.30       211
       insult       0.43      0.77      0.55      3427
identity_hate       0.41      0.53      0.46       712

    micro avg       0.45      0.79      0.57     14498
    macro avg       0.38      0.67      0.48     14498
 weighted avg       0.45      0.79      0.57     14498
  samples avg       0.07      0.07      0.0

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
